# **Project title:** Quantitative assessment of slc35a2 localization to the Golgi in single cells
### **Researcher:** Xavier Williams
### **Notebook author:** Shannon Rhoads (srhoads@unc.edu)

### **Editing & version log:**
- v0.1: 2026 - downloaded [full_quantification_pipeline notebook](https://github.com/SCohenLab/infer-subc/blob/main/notebooks/part_2_quantification/full_quantification_pipeline.ipynb) from [infer-subc version v1.0.0](https://github.com/SCohenLab/infer-subc/tree/v1.0.0)
- v0.2: 2026 - edits were made to do the following
    - implement an optional per cell analysis that can be used if there is a nucleus and cell mask
    - pair down the analysis to quantify the intensity of the raw image if only an organelle is included wihtout cell and nucleus masks

----------
## **How to use Jupyter notebooks**

Advance through each block of code below by pressing `Shift`+`Enter` or pressing the "Execute Cell" (`▶️`) button to the left of each block.

You will see a series of instructions before each block of code. Be on the look out for the following headers and follow the instructions accordingly:
- &#x1F3C3; **Run code; no user input required** - proceed without adding anything to the code block
- &#x1F453; **FYI** (for your information) - helpful information usually to bring context to what is going on
- &#x1F6D1; &#x270D; **User Input Required** - stop and input the appropriate information about your data. The following indicator will also be present in the code block:
   ```python 
   #### USER INPUT REQUIRED ###
   ```

----------

# **Notebook title:** Batch Process Quantification

***Prior to this notebook, you should have already segmented the Golgi (and cell/nuc masks if including) for one or more datasets.***

### **Purpose**
In this notebook, the raw image and segmentations are imported to quantify the intensity of fluorescent labels from the raw image inside the specified organelle, here the Golgi, within an entire image. Additionally, segmentations of proteins of interest (corresponding to one of the intensity channesl) can be included to quantify Mander's correlation coefficiency and portion of the segmented protein that resides in the specified organelles ("volume fraction"). 

If a cell and nucleus mask are included, the analysis can expand to quantify the size, shape, intensity, and distribution of the specified organell within the cell as well. 


### ➡️ **Input**
The following files and information will be necessary for batch processing segmentations:
1. The **raw confocal microscopy images** (".tiff", ".tif", or ".czi") from one experimental replicate, where each channel of the image represents one of the organelles or structures being segmented. The pipeline will only accept a single file path as the input for the raw data and all images of the same type within that folder will be processed.
2. The **segementation images** (".tiff" or ".tif") from one experimental replicate. The names of these files should match the corresponding raw data file name except that they include a suffix indicating which object was segmented (e.g., "{original-image-name}-golgi.tif").
3. The **output file location** where the resulting quantification files will be saved.


### **Output** ➡️
One or more .csv files containing the quantitative output from all the images in the dataset. If a cell and nucleus mask are NOT included, one data table will be saved. Each row of data represents the intensity values inside the specified organelle for a single image. If the cell and nucleus mask are included, there will be multiple datatables (one for morphology/intensity measures and one for distribution measures). Each row in these tables will represent each individual organelle object identified during segmentation; the associated cell is named in one of the columns. To summarize this data per cell, a secondary summary step is included below.

> #### **Example file structure:**
>
> - 📂 **Experiment1**
>   - 📂 raw-data               ```<-- Input directory```
>       - 🗋 "image1.tiff"
>       - 🗋 ...
>   - 📂 segmentation-output               ```<-- Input directory```
>       - 🗋 "image1-masks.tiff"
>       - 🗋 "image1-golgi.tiff"
>       - 🗋 ...
>   - 📂 quantifcation-output               ```<-- Output directory```
> - 📂 **Experiment2**
>   - 📂 ...
>
-----

### 👣 **Summary of steps**  

➡️ **BATCH PROCESSING**
- **`Step 1`** - Define the paths to your data and desired output folder.

- **`Step 2`** - Run `batch_process_quantification()`
    - **Input**: Raw & segmentation images
    - **Output**: CSV files

    This step outputs per image intensity measurements, if no masks are included OR

________________
## 	**IMPORTS**

#### &#x1F6D1; &#x270D; **User Input Required:**

Please specify the following information about your data: 
- `path_to_repository`: the path to the local golgi-coloc-analysis repository folder as a string

In [2]:
#### USER INPUT REQUIRED ###
path_to_repository = r"C:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis"

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block loads all of the necessary python packages and functions you will need for this notebook. Additionally, a [Napari](https://napari.org/stable/) window will open; this is where you will be able to visual the segmentations.

In [3]:
from pathlib import Path
import pandas as pd
import os

import napari

from infer_subc.core.file_io import (read_czi_image,
                                        export_inferred_organelle,
                                        import_inferred_organelle,
                                        list_image_files,)

from infer_subc.core.img import *

from typing import List, Union, Dict
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image
import numpy as np
import time
from infer_subc.utils.stats import (get_contact_metrics_3D, get_org_morphology_3D, get_XY_distribution, 
                                    get_Z_distribution, get_region_morphology_3D)
import itertools 

from bioio import BioImage

from skimage.measure import manders_coloc_coeff

import sys
sys.path.insert(0, path_to_repository)
from src.quantification import batch_process_intensity_quant, batch_process_quantification, batch_summary_stats, batch_intensity_coloc_quant

%load_ext autoreload
%autoreload 2

__________________________
## **WHOLE IMAGE INTENSITY QUANTIFICATION**

*The pipeline below is to be used if your analysis includes a segmented organelle of interest, BUT DOES NOTE INCLUDE cell and nuclei masks.*

#### **Per image intensity quantification:**
This pipeline will quantify the total (sum) intensity of each channel in the raw image within your segmented objects (as a whole, not individual objects) and in the whole image. These metrics can be used to find the percentage of the signal inside the segmented object of interest. You can include one or more segmented obejcts per image and the intensity measure will be carried out for each.

#### &#x1F6D1; &#x270D; **User Input:**

**Function Parameters:**
- *out_file_name*: The prefix to use when naming the output datatables
- *seg_path*: Path or str to the folder that contains the segmentation tiff files
- *out_path*: Path or str to the folder that the output datatables will be saved to
- *raw_path*: Path or str to the folder that contains the raw image files
- *raw_file_type*: The file type of the raw data; ex - ".tiff", ".czi"
- *raw_chan_axis*: Index of the channel axis in the raw intensity image
- *raw_Z_axis*: Index of the Z axis in the raw intensity image
- *raw_Y_axis*: Index of the Y axis in the raw intensity image
- *raw_X_axis*: Index of the X axis in the raw intensity image
- *organelle_names*: A list of the organelles that will be used for intensity measurements (portion of intensity in image that exists inside each organelle) and colocalization analysis. Organelle names listed here should have a corresponding segmentation image (i.e., if "golgi" is listed here, there should be a segmentation file with the suffix "-golgi.tiff").
- *proteins_of_interest*: A list of the proteins that will be used for colocalization analysis; the manders colocalization coefficient of the proteins intensity inside the specified organellesand portion of the protein volume in the organelle segmentations will be calculated. This is optional; specify None as the input if you do not want to calculate these colocalization metrics. Protein names listed here should have a corresponding entry in the channel_dict and segmentation image (i.e., if "proteinA" is listed here, there should be a channel_dict entry like {0: "proteinA"} and a segmentation file with the suffix "-proteinA.tiff").   
- *channel_dict*: A dictionary mapping channel indices (keys) to their names (values) in the raw image. Only channels specified in this dictionary will be used for intensity measurements. They will be identified as the associated keys in the output datatables.
- *seg_suffix*: Any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix; this was specified during batch processing, if it was used. If not used, indicate None.


#### &#x1F6D1; &#x270D; **User Input Required:**

In [4]:
#### USER INPUT REQUIRED ###
out_file_name = "test_quantification_redo1"
seg_path = Path(os.path.dirname(os.getcwd()))  / "test-segmentation/single"
out_path = Path(os.path.dirname(os.getcwd()))  / "test-quantification/single" 
raw_path = Path(os.path.dirname(os.getcwd()))  / "pilot-data/single"
raw_file_type = ".czi"
raw_chan_axis = 0
raw_Z_axis = 1
raw_X_axis = 3
raw_Y_axis = 2
organelle_names= ['golgi', 'slc35a2']
proteins_of_interest = ['slc35a2']
channel_dict = {0: 'golgi',
                1: 'slc35a2',
                2: 'DAPI'}
seg_suffix = None

#### &#x1F3C3; **Run code; no user input required**

In [5]:
out_tab = batch_intensity_coloc_quant(out_file_name = out_file_name,
                                      seg_path = seg_path,
                                      out_path = out_path, 
                                      raw_path = raw_path, 
                                      raw_file_type = raw_file_type, 
                                      raw_chan_axis = raw_chan_axis,
                                      raw_Z_axis = raw_Z_axis,
                                      raw_X_axis = raw_X_axis,
                                      raw_Y_axis = raw_Y_axis,
                                      organelle_names= organelle_names,
                                      proteins_of_interest = proteins_of_interest,
                                      channel_dict = channel_dict,
                                      seg_suffix = seg_suffix)

out_tab

Processing image 03022026R55LB1T1P1-01...
Completed processing for 1 images in 0.2587908148765564 mins.
Quantification for 1 files is COMPLETE! Files saved to 'c:\Users\Shannon\Documents\Python_Scripts\Golgi-coloc-analysis\test-quantification\single'.
It took 0.2592881162961324 minutes to quantify these files.


,golgi-ch_intensity-sum_in-whole-image,golgi-ch_intensity-sum_in-golgi,golgi-ch_intensity-sum_in-slc35a2,slc35a2-ch_intensity-sum_in-whole-image,slc35a2-ch_intensity-sum_in-golgi,slc35a2-ch_intensity-sum_in-slc35a2,DAPI-ch_intensity-sum_in-whole-image,DAPI-ch_intensity-sum_in-golgi,DAPI-ch_intensity-sum_in-slc35a2,portion-golgi-ch-intensity_in-golgi,portion-golgi-ch-intensity_in-slc35a2,portion-slc35a2-ch-intensity_in-golgi,portion-slc35a2-ch-intensity_in-slc35a2,portion-DAPI-ch-intensity_in-golgi,portion-DAPI-ch-intensity_in-slc35a2,manders-slc35a2_in-golgi,volume-fraction-slc35a2_in-golgi,manders-slc35a2_in-slc35a2,volume-fraction-slc35a2_in-slc35a2
file_name,,,,,,,,,,,,,,,,,,,
03022026R55LB1T1P1-01,1.355234e+11,9.318143e+09,270157922.0,5.646614e+10,236957628.0,975127328.0,4.253589e+11,1.671193e+09,484861149.0,0.068757,0.001993,0.004196,0.017269,0.003929,0.00114,0.007837,0.009261,1.0,1.0


__________________________
## **SINGLE CELL QUANTIFICATION**

*The pipeline below is to be used if your analysis includes 1) a segmented organelle of intereste and 2) segmented cell and nuclei masks.*

#### **Batch process quantification:**
After all segmentations for all organelles and masks are correctly assembled into one folder per experiment, you can begin quantification. The output will include ***`per object`*** measurements. Use the function below to input the correct folder paths to your data and settings you'd like to use for analysis.

> ***Note:** The logic to the following two notebooks is for `batch_process_quantification()` to be run on each experimental replicate (i.e., group of images collected on the same data from the same sample preparation) individually. Then, `batch_summary_stats()` summarizes all of the data together in a "per cell" fashion. The first column in the output folders will specify the experimental replicate for each image.*

#### &#x1F6D1; &#x270D; **User Input:**

**Function Parameters:**
- *out_file_name*: str;
    the prefix to use when naming the output datatables
- *seg_path*: Union[Path,str];
    Path or str to the folder that contains the segmentation files
- *out_path*: Union[Path, str];
    Path or str to the folder that the output datatables will be saved to
- *raw_path*: Union[Path,str];
    Path or str to the folder that contains the raw image files that were used to creat the segmentations
- *raw_file_type*: str;
    the file type of the raw data; ex - ".tiff", ".czi"
- *organelle_names*: List[str];
    a list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files
    Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
- *organelle_channels*: List[int];
    a list of channel indices associated to respective organelle staining in the raw image; the indices should listed in same order in which the respective segmentation name is listed in organelle_names
- *region_names*: List[str];
    a list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
- *masks_file_name*: List[str];
    the suffix of the "masks" segmentation file; ex- ["cell", "nuc"] or ["cell"]
- *mask*: str;
    the name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
- *dist_centering_obj*: str;
    the name of the region or object to use as the centering object in the get_XY_distribution function
- *dist_num_bins*: int;
    the number of bins for the get_XY_distribution function
- *dist_center_on*: bool=False;
    for get_XY_distribution:
    True = distribute the bins from the center of the centering object
    False = distribute the bins from the edge of the centering object
- *dist_keep_center_as_bin*: bool=True;
    for get_XY_distribution:
    True = include the centering object area when creating the bins
    False = do not include the centering object area when creating the bins
- *dist_zernike_degrees*: Union[int, None]=None;
    for get_XY_distribution:
    the number of zernike degrees to include for the zernike shape descriptors; if None, the zernike measurements will not 
    be included in the output
- *include_contact_dist*:bool=True;
    whether to include the distribution of contact sites in get_contact_metrics_3d(); True = include contact distribution
- *scale*:bool=True;
    True indicates you will use "real-world" scaling units (e.g., um, nm, etc. based on the image metadata)
- *seg_suffix*:Union[str, None]=None;
    any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix

#### &#x1F6D1; &#x270D; **User Input Required:**

In [ ]:
#### USER INPUT REQUIRED ###
out_file_name = ""
seg_path = ""
out_path = ""
raw_path = ""
raw_file_type = ".czi"
raw_chan_axis = 1
raw_Z_axis = 0
raw_Y_axis = 2
raw_X_axis = 3
organelle_names = ['golgi']
organelle_channels = [0]
region_names = ['nuc', 'cell']
masks_file_name = ['nuc', 'cell']
mask = 'cell' 
dist_centering_obj = 'nuc'
dist_num_bins = 5 
dist_center_on = False
dist_keep_center_as_bin = True
dist_zernike_degrees = None
include_contact_dist = True
scale = True
seg_suffix = "-"

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
seg=batch_process_quantification(out_file_name=out_file_name,
                                  seg_path=seg_path,
                                  out_path = out_path, 
                                  raw_path=raw_path,
                                  raw_file_type = raw_file_type,
                                  raw_chan_axis=raw_chan_axis,
                                  raw_Z_axis=raw_Z_axis,
                                  raw_Y_axis=raw_Y_axis,
                                  raw_X_axis=raw_X_axis,
                                  organelle_names = organelle_names,
                                  organelle_channels= organelle_channels,
                                  region_names= region_names,
                                  masks_file_name= masks_file_name,
                                  mask= mask,
                                  dist_centering_obj= dist_centering_obj,
                                  dist_num_bins= dist_num_bins,
                                  dist_center_on= dist_center_on,
                                  dist_keep_center_as_bin= dist_keep_center_as_bin,
                                  dist_zernike_degrees= dist_zernike_degrees,
                                  include_contact_dist= include_contact_dist,
                                  scale=scale,
                                  seg_suffix=seg_suffix)

#### **Summarize data per cell:**
The function below takes in the quantification results output from `batch_process_quantification()` from one or more experimental replicates. The results will be aggregated and summarized per cell.

**Function Parameters:**
- *csv_path_list*: List[str];
    A list of path strings where .csv files output from batch_process_quantification() function are located.
- *out_path*: str;
    A path string where the summary data files will be saved.
- *out_preffix*: str;
    The prefix used to name the output file.

#### &#x1F6D1; &#x270D; **User Input Required:**

In [ ]:
#### USER INPUT REQUIRED ###
csv_path_list= [out_path]
out_path = out_path
out_preffix = "20251216_test-multi-cell_"

#### &#x1F3C3; **Run code; no user input required**

In [ ]:
out=batch_summary_stats(csv_path_list=csv_path_list,
                         out_path= out_path,
                         out_preffix=out_preffix)

# Congratulations!! 🎉
### You've made it through `infer-subc`. The next step is to analyze your data as you see fit. The definitions for each metric are included [here](./batch_summary_stats_output_definitions.xlsx). Happy analyzing!!